1. 路由

In [3]:
from fastapi import FastAPI

# 创建 FastAPI 实例
app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello"}

@app.get("/hello")
async def get_hello():
    return {"message": "world"}

In [4]:
import threading
import time
import uvicorn

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# Jupyter 自己已经有事件循环，这里把 Uvicorn 放到后台线程里启动
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [13224]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started at http://127.0.0.1:8000


In [5]:
from urllib.request import urlopen
import json

response = urlopen("http://127.0.0.1:8000/hello")
json.loads(response.read().decode("utf-8"))

INFO:     127.0.0.1:52748 - "GET /hello HTTP/1.1" 200 OK


{'message': 'world'}

In [6]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [13224]


Server stopped


2. 路径参数 Path + 类型注释

In [8]:
from fastapi import FastAPI, Path

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello"}

@app.get("/book/{id}")
async def get_book(id: int = Path(..., gt=0, lt=101, description="图片编号1-100")):
    return {"id": id}

@app.get("/zuthor/{name}")
async def get_name(name: str = Path(..., min_length=2, max_length=10)):
    return {"name": name}

import threading
import time
import uvicorn

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [36338]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started at http://127.0.0.1:8000


In [9]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36338]


Server stopped


3. 查询参数 Query + 类型注释

In [10]:
from fastapi import FastAPI, Query

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello world"}

@app.get("/news/news_list")
async def get_news_list(
        skip: int = Query(0, description="skipped record", lt=100),
        limit: int = Query(10, description="limit record")
):
    return {"skip": skip, "limit": limit}

import threading
import time
import uvicorn
import urllib.request
import urllib.error

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)  # server.run是一个方法
thread.start()

# 轮询检测服务器是否就绪（最多等5秒）
max_retries = 50  # 最多尝试50次
for i in range(max_retries):
    try:
        # 尝试发送一个测试请求
        urllib.request.urlopen("http://127.0.0.1:8000")
        print("Server ready")
        break
    except urllib.error.URLError:
        # 服务器还没准备好，等待0.1秒后重试
        time.sleep(0.1)
else:
    # 循环正常结束（没有被break），说明超时了
    print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [36338]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55848 - "GET / HTTP/1.1" 200 OK
Server ready


In [11]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36338]


Server stopped


4. 请求体参数

In [13]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
import threading
import time
import uvicorn

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello world"}

# 定义一个模型类
class User(BaseModel):
    username: str = Field(default="xiaowang", min_length=2, max_length=10, description="username")
    password: str = Field(min_length=3, max_length=20)
    
@app.post("/register")
async def register(user: User):
    return user

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)  # server.run是一个方法
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [36338]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started at http://127.0.0.1:8000


In [14]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36338]


Server stopped


5. 响应类型-HTML格式

In [15]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import threading
import time
import uvicorn

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello world"}

# 接口 -> HTML 代码
@app.get("/html", response_class=HTMLResponse)
async def get_html():
    return "<h1>hello world</h1>"

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [36338]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started at http://127.0.0.1:8000


In [16]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36338]


Server stopped


6. 响应类型-文件格式

In [ ]:
from fastapi import FastAPI
from fastapi.responses import FileResponse
import threading
import time
import uvicorn

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello world"}

# 接口： 返回一张图片内容
@app.get("/file")
# @app.get("/file", response_class=FileResponse)
async def get_file():
    path = "./hello.png"
    return FileResponse(path)
    # return path

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

In [ ]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

7. 响应类型-流式格式

In [3]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from PIL import Image
from io import BytesIO
import random
import threading
import time
import uvicorn

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello world"}

# 接口： 返回一张随机 RGBA 图片内容
@app.get("/file")
async def get_file():
    width, height = 256, 256
    color = (
        random.randint(0, 255),
        random.randint(0, 255),
        random.randint(0, 255),
        random.randint(0, 255),
    )
    image = Image.new("RGBA", (width, height), color)

    buffer = BytesIO()
    image.save(buffer, format="PNG")
    buffer.seek(0)

    return StreamingResponse(buffer, media_type="image/png")

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [58844]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started at http://127.0.0.1:8000


In [4]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [58844]


Server stopped


8. 自定义响应数据格式

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import threading
import time
import uvicorn

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "hello world"}

# 需求：新闻接口 → 响应数据格式 id、title、content
class News(BaseModel):
    id: int
    title: str
    content: str
    
@app.get("/news/{id}", response_model=News)
async def get_news(id: int):
    id_list = [1, 2, 3, 4, 5]
    if id not in id_list:
        raise HTTPException(status_code=404, detail="news not found")
    return {"id": id, "title": f"this is {id}", "content": "hello world"}

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1)
print("Server started at http://127.0.0.1:8000")

INFO:     Started server process [56902]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server started at http://127.0.0.1:8000


In [2]:
server.should_exit = True
thread.join(timeout=3)
print("Server stopped")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [56902]


Server stopped
